# Lab 6: Recurrent Neural Networks (RNNs) in PyTorch

Objectives: By the end of this lab, students should be able to:
(1) Understand sequence data and RNN intuition, (2) Implement an RNN using PyTorch, 
(3) Train an RNN for sequence classification, and (4) Evaluate and improve model performance

## Task Overview:
    You will build a model that classifies sentences as positive or negative sentiment using an RNN.

### Part 1: Dataset Preparation
We will use a toy dataset (for simplicity).

In [5]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

# Data
sentences = [
"i love this movie",
"this film is great",
"amazing experience",
"i hate this movie",
"this film is terrible",
"bad experience"
]

labels = torch.tensor([1, 1, 1, 0, 0, 0])

# Tokenization
tokenized = [s.split() for s in sentences]

# Build vocab
vocab = {"<pad>": 0}
for sentence in tokenized:
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

# Convert to indices
indexed = [
torch.tensor([vocab[word] for word in sentence])
for sentence in tokenized
]

# Pad sequences
padded = pad_sequence(indexed, batch_first=True, padding_value=0)

print("Vocab:", vocab)
print("Padded shape:", padded.shape)

Vocab: {'<pad>': 0, 'i': 1, 'love': 2, 'this': 3, 'movie': 4, 'film': 5, 'is': 6, 'great': 7, 'amazing': 8, 'experience': 9, 'hate': 10, 'terrible': 11, 'bad': 12}
Padded shape: torch.Size([6, 4])


### Part 2: Build the RNN Model

Implement an RNN classifier using:

Embedding layer

RNN layer

Fully connected layer


In [9]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        out = self.fc(hidden.squeeze(0))
        return out

# Hyperparameters
vocab_size = len(vocab)
embed_dim = 10
hidden_dim = 16
output_dim = 2

model = RNNClassifier(vocab_size, embed_dim, hidden_dim, output_dim)


### Part 3: Training

Train the model using:

Loss: CrossEntropyLoss

Optimizer: Adam

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 100

for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(padded)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 0.6565
Epoch 10, Loss: 0.2215
Epoch 20, Loss: 0.0077
Epoch 30, Loss: 0.0014
Epoch 40, Loss: 0.0006
Epoch 50, Loss: 0.0004
Epoch 60, Loss: 0.0003
Epoch 70, Loss: 0.0003
Epoch 80, Loss: 0.0002
Epoch 90, Loss: 0.0002


### Part 4: Evaluation

Evaluate accuracy on the training set.

In [11]:
with torch.no_grad():
    outputs = model(padded)
    predictions = torch.argmax(outputs, dim=1)
    accuracy = (predictions == labels).float().mean()

print("Predictions:", predictions)
print("Accuracy:", accuracy.item())

Predictions: tensor([1, 1, 1, 0, 0, 0])
Accuracy: 1.0


### Part 5: 

Replace the above RNN with LSTM.

self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

And modify forward:

output, (hidden, cell) = self.rnn(embedded)
out = self.fc(hidden.squeeze(0))
